# Replikasi Parsial: *Gradient-Boosted Tree* untuk Prediksi Penyakit Kardiovaskular dengan Apache Spark (PySpark MLlib)


---




### **Nama : Atika Adelia**

### **NIM : 123140172**


---



## Artikel acuan

Gupta, Y. K., & Kumari, S. (2021). Performance evaluation of distributed machine learning for cardiovascular disease prediction in Spark. *2021 5th International Conference on Trends in Electronics and Informatics (ICOEI)*, 1506–1512. IEEE. https://doi.org/10.1109/ICOEI51242.2021.9452955

**Dataset:** Ulianova, S. (n.d.). *Cardiovascular Disease dataset* [Data set]. Kaggle. https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset

## Tujuan

Mengulang bagian inti artikel, yaitu membangun pengklasifikasi **GBT** (`GBTClassifier`) pada Spark MLlib untuk memprediksi ada/tidaknya penyakit kardiovaskular (`cardio` = 1/0), lalu membandingkan hasilnya (AUC-ROC, *accuracy*, *precision*, *recall*, F1) dengan angka yang dilaporkan artikel. Empat model lain pada artikel (LR, DT, SVM, RF) **tidak** direplikasi.



## 1. Persiapan lingkungan

Artikel memakai Spark 2.4.7. Versi itu tidak kompatibel dengan Python dan Java pada Colab saat ini, sehingga replikasi ini memakai **PySpark 3.5.x** + **Java 17**. API `GBTClassifier`, `CrossValidator`, 

In [32]:
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import reduce
from IPython.display import display

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import roc_curve

In [29]:
# 1.1 Pasang Java 17 dan pustaka Python (cukup sekali per sesi Colab)
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1
!pip install -q "pyspark>=3.5,<4" kagglehub

In [30]:
# 1.2 Atur JAVA_HOME dan catat versi seluruh komponen (syarat reprodusibilitas, panduan 4.2)
import os, glob, sys, subprocess

java_dirs = sorted(glob.glob("/usr/lib/jvm/java-17-openjdk-*"))
if not java_dirs:
    raise RuntimeError("Java 17 tidak ditemukan. Jalankan ulang sel instalasi (1.1).")
os.environ["JAVA_HOME"] = java_dirs[0]
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

import numpy, pandas, matplotlib, sklearn, pyspark

java_ver = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0]
print(f"PySpark      : {pyspark.__version__}")

PySpark      : 3.5.9
